# Querying SQL Data with Python

In the previous chapter, you saw how to use the [Python DB-API to upload CSV data](2.1-load-csv-data.ipynb) into InterSystems IRIS For Health via SQL `CREATE` and `INSERT` statements. 

In this brief tutorial, you will see how very similar code can be used to query the data we uploaded. 

## Create Connection 

The code here is the same as shown in the previous tutorial:


In [1]:
import iris
connection_args = { 
    "hostname":"localhost", 
    "port": 32782, 
    "namespace": "FHIRSERVER", 
    "username": "SuperUser",
    "password" : "SYS"
}
conn = iris.connect(**connection_args)
cursor = conn.cursor() 

## Query Database

Querying the database is a three step process: 
- Define Query
- Execute Query
- Fetch Results


### Executing query
Let's see the first two in action with the "Sample.Person" table created in the [load csv](2.1-load-csv-data.ipynb) tutorial 


In [2]:
sql_query = """SELECT 
FirstName, LastName, Age
FROM Sample.Person"""

out = cursor.execute(sql_query)

A common mistake here is assuming that the output is returned as an object somehow, or alternatively that the returned object represents a status value where a negative value represents failure. 

A successful output here returns -1, whereas an unsuccessful output exits with an error. 

In [3]:
print(out)

-1


Instead of returning the results, the cursor object keeps track of the position in the database and then fetches individual rows of the table:

In [4]:
first_row = cursor.fetchone()

print(first_row)

The row is returned as a DataRow object, but it can easily be converted to a list for easier access. 

In [5]:
print(list(first_row))

['Peter', 'Parker', 18]


### Fetching Multiple

You can fetch the next n rows with `fetchmany()` or fetch all remaining rows using `fetchall()`:

In [6]:
next_three_rows = cursor.fetchmany(3)
print(next_three_rows)

(('Bruce', 'Wayne', 50), ('Natasha', 'Romanoff', 30), ('Clark', 'Kent', 35))


> Important! the cursor always fetches the next row or rows, so to ensure you get all data, re-execute the query

In [7]:
cursor.execute(sql_query) 
all_rows = cursor.fetchall()
print(all_rows)

(('Peter', 'Parker', 18), ('Bruce', 'Wayne', 50), ('Natasha', 'Romanoff', 30), ('Clark', 'Kent', 35), ('Diana', 'Prince', 32), ('Peter', 'Parker', 25), ('Tony', 'Stark', 45), ('Steve', 'Rogers', 39), ('Carol', 'Danvers', 34), ('Matt', 'Murdock', 31), ('Barry', 'Allen', 28))


All rows is now a tuple of tuples (tuples are immutable lists) so you can iterate through the results easily: 

In [8]:
for row in all_rows: 
    print(f"First Name: {row[0]}, Last Name: {row[1]}, Age: {row[2]}")

First Name: Peter, Last Name: Parker, Age: 18
First Name: Bruce, Last Name: Wayne, Age: 50
First Name: Natasha, Last Name: Romanoff, Age: 30
First Name: Clark, Last Name: Kent, Age: 35
First Name: Diana, Last Name: Prince, Age: 32
First Name: Peter, Last Name: Parker, Age: 25
First Name: Tony, Last Name: Stark, Age: 45
First Name: Steve, Last Name: Rogers, Age: 39
First Name: Carol, Last Name: Danvers, Age: 34
First Name: Matt, Last Name: Murdock, Age: 31
First Name: Barry, Last Name: Allen, Age: 28


## Adding parameters

If you need to execute queries with parameters, you should add these to the query with "?" placeholders rather than using string comprehension — this provides basic sanitization to prevent SQL injection. You can see this as follows: 

In [9]:
age_query = "SELECT FirstName, LastName, Age FROM Sample.Person WHERE Age = ?" 
max_age = 50
cursor.execute(age_query, [max_age])
cursor.fetchall()

(('Bruce', 'Wayne', 50),)

## Close Connection

Remember to close the connection to avoid running into license constraints. InterSystems IRIS Community supports a limited amount of active connections before it rejects them with license errors. 

In [10]:
conn.close()